# Content-Based Recommendation System

In [17]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:\DATA SCIENCE\Projects\Capstone_Project\Datasets\cleaned_amazon_audio.csv")

In [2]:
df.columns

Index(['asin', 'product_title', 'brand', 'price', 'discount_percent', 'rating',
       'review_count', 'category', 'product_link', 'search_term',
       'value_score', 'device_type', 'is_wired', 'has_mic'],
      dtype='object')

### Combining all text together

In [18]:
df['combined_text'] = (
    df['product_title'] + " " +
    df['brand'] + " " +
    df['category'] + " " +
    df['device_type']
)

### NLP

In [13]:
# Tokenization,stopwords removal,encoding
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english', max_features=3000)
text_matrix = tfidf.fit_transform(df['combined_text'])

### Appying Cosine Similarity

In [14]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_from_text(query, top_n=5, threshold=0.2):
    # Convert user query to vector
    query_vec = tfidf.transform([query])
    
    # Compute similarity
    sim_scores = cosine_similarity(query_vec, text_matrix).flatten()

    # Get sorted indices (highest similarity first)
    top_indices = sim_scores.argsort()[::-1]

    # Filter relevant results
    filtered = [i for i in top_indices if sim_scores[i] > threshold]

    # If nothing found
    if len(filtered) == 0:
        return None   # ✅ IMPORTANT FIX

    # Return top results
    return df[['product_title', 'price', 'rating', 'is_wired']].iloc[filtered[:top_n]]

In [15]:
recommend_from_text(" wireless " )

,product_title,price,rating,is_wired
200,wireless headset sh12 wireless bluetooth headp...,649.0,2.9,0
1258,compatible for wireless earbuds with charging ...,499.0,5.0,0
768,wireless bluetooth neckband with bluetooth 54 ...,427.0,4.1,0
754,wireless headphones neckband earphones in ear ...,499.0,5.0,0
1091,wireless earbudsbluetooth 54 headphones in ear...,6432.0,4.9,0


In [16]:
import pickle

pickle.dump(tfidf, open("tfidf.pkl", "wb"))
pickle.dump(text_matrix, open("matrix.pkl", "wb"))
df.to_pickle("data.pkl")